In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

soda_path = r'C:\Users\jiuwe\study\data\soda\\'
noaa_path = r'C:\Users\jiuwe\study\data\noaa-20c\\'

In [ ]:
ds_t = xr.open_dataset(soda_path+'soda.temp.nc')
ds_u = xr.open_dataset(soda_path+'soda.ucur.nc')
ds_v = xr.open_dataset(soda_path+'soda.vcur.nc')
ds_w = xr.open_dataset(soda_path+'soda.wcur.nc')
ds_us= xr.open_dataset(soda_path+'soda.ustr.nc')
ds_vs= xr.open_dataset(soda_path+'soda.vstr.nc')
ds_ssh=xr.open_dataset(soda_path+'soda.ssh.nc')
ds_pre=xr.open_dataset(noaa_path+'prmsl.mon.mean.tp.nc')

temp = ds_t.TEMP
ucur = ds_u.U
vcur = ds_v.V
wcur = ds_w.W
taux = ds_us.TAUX
tauy = ds_vs.TAUY
ssh  = ds_ssh.SSH
pre = ds_pre.prmsl

level = [5.01, 15.07, 25.28, 35.76, 46.61, 57.98, 70.02, 82.92, 96.92, 112.32, 129.49, 148.96, 171.4, 197.79 ]

temp = temp.assign_coords(level=level).rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))
ucur = ucur.assign_coords(level=level).rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))
vcur = vcur.assign_coords(level=level).rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))
wcur = wcur.assign_coords(level=level).rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))

taux = taux.rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))
tauy = tauy.rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))
ssh  = ssh.rename({'TIME':'time'}).assign_coords(time=pd.date_range('1900-01', '2010-12', freq='MS'))

# rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()
rolling = lambda x:x.mean(dim='time').broadcast_like(x)
T = temp.loc[:, 0:20, :, :].mean(dim='level')
ML_T = T.groupby('time.month').map(rolling)
ML_t = T - ML_T

U_ = ucur.loc[:, 0:20, :, :].mean(dim='level')
U = U_.groupby('time.month').map(rolling)
u = U_ - U

V_ = vcur.loc[:, 0:20, :, :].mean(dim='level')
V = V_.groupby('time.month').map(rolling)
v = V_ - V

ML_w = wcur.loc[:, 0:20, :, :].mean(dim='level')
W = ML_w.groupby('time.month').map(rolling)
w = ML_w - W

TAUX = taux.groupby('time.month').map(rolling)
taux = taux - TAUX
TAUY = tauy.groupby('time.month').map(rolling)
tauy = tauy - TAUY

SSH = ssh.groupby('time.month').map(rolling)
ssh = ssh - SSH

P = pre.groupby('time.month').map(rolling)
p = pre - P

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2009-01')]
ep = ep[ep < np.datetime64('2009-01')]
mix = mix[mix < np.datetime64('2009-01')]
suc = suc[suc < np.datetime64('2009-01')]
events = [mix, ep, cp, suc]

In [ ]:
lon = np.arange(120, 281)
lat = np.arange(-40, 41)
month = np.arange(-6, 22)
ds_nino = xr.Dataset(
    {
        't':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'T':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'u':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'U':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'v':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'V':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'taux':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'tauy':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'w':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'W':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'ssh':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        'pre':(['type', 'month', 'lat', 'lon'], np.empty((4, len(month), len(lat),len(lon)))),
        
    },
    coords={
        'type':(['type'], ['MIX', 'EP', 'CP', 'SUC']),
        'month':(['month'], month),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon),
    }
)

for i in range(4):
    event = events[i]
    for j in range(len(month)):
        ds_nino.t[i, j, :, :] = ML_t.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.T[i, j, :, :] = ML_T.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.u[i, j, :, :] = u.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.U[i, j, :, :] = U.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.v[i, j, :, :] = v.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.V[i, j, :, :] = V.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.w[i, j, :, :] = w.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.W[i, j, :, :] = W.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.taux[i, j, :, :] = taux.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.tauy[i, j, :, :] = tauy.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.ssh[i, j, :, :] = ssh.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')
        ds_nino.pre[i, j, :, :] = p.loc[event + np.timedelta64(month[j], 'M'), :, :].mean(dim='time')

# ds_nino.to_netcdf('./data-20c/composite-nino.nc')

In [ ]:
ds_nino = xr.open_dataset('./data-20c/composite-nino.nc')

In [ ]:
fig, axs = plt.subplots(4, 3, figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=135)})
month = ds_nino.month.values
lon = ds_nino.lon.values
lat = ds_nino.lat.values

# data = ds_nino.pre
# U = ds_nino.taux
# V = ds_nino.tauy

data = ds_nino.t.rolling(month=3).mean()
U = ds_nino.taux.rolling(month=3).mean()
V = ds_nino.tauy.rolling(month=3).mean()
for i, ax in enumerate(axs.flatten()):
    mon = month[i+3]
    tp = 'MIX'
    ax.set_title(f'{mon}')
    ax.set_aspect('auto')
    ax.set_extent([120, 280, -40, 40], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#F5F5F5', lw=0.1, zorder=1)
    ax.coastlines(lw=0.8, color='k')
    ax.set_yticks(np.arange(-40, 41, 20), crs=ccrs.PlateCarree())
    ax.set_xticks(np.arange(120, 281, 40), crs=ccrs.PlateCarree())   
    ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    cb = ax.contourf(lon, lat, data.loc[tp, mon], levels=np.linspace(-2, 2, 21), cmap='bwr', transform=ccrs.PlateCarree(), extend='both')
    # cb = ax.contourf(lon, lat, ds_nino.ssh.loc[tp, mon].rolling(lat=4, lon=4, center=True).mean(), levels=np.linspace(-4, 4, 21)/16, cmap='bwr', transform=ccrs.PlateCarree(), extend='both')
    # u = ds_nino.taux.rolling(month=3).mean().loc[tp, mon].values
    # v = ds_nino.tauy.rolling(month=3).mean().loc[tp, mon].values
    u = U.loc[tp, mon].values
    v = V.loc[tp, mon].values
    skip = 6
    ax.quiver(lon[::skip], lat[::skip], u[::skip, ::skip], v[::skip, ::skip], 
              scale=0.35, width=0.008, headwidth=2.5, headlength=3, headaxislength=3.5, alpha=0.7, transform=ccrs.PlateCarree())
    
cbaxes = fig.add_axes([0.9, 0.2, 0.01, 0.6])
cb = plt.colorbar(cb, cax=cbaxes, orientation="vertical")
fig.subplots_adjust(left=0.07, right=0.85, bottom=0.05, top=0.95, wspace=0.4, hspace=0.4) 

In [ ]:
ds_nino.taux.loc['CP', month, -20:20, :].plot.contourf(col='month', col_wrap=4, levels=21, cmap='bwr', size=2, aspect=2)

In [ ]:
ds_nino.taux.loc[:, month, -5:5, :].rolling(month=3).mean().mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr', size=2, aspect=1.4)

In [ ]:
ds_nino.pre.loc[:, month, -5:5, :].rolling(month=3).mean().mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr', size=2, aspect=1.4)

In [ ]:
ds_nino.t.loc[:, month, -5:5, :].rolling(month=3).mean().mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr', size=2, aspect=1.4)

In [ ]:
lons = np.arange(200, 230)
lats = 20/30*(lons-200)
ds_nino.pre[:, :].interp(lat=lats, lon=lons).sel(lat=xr.DataArray(lats, coords={'points':lats}), lon=xr.DataArray(lons, coords={'points':lats})).plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr')

In [ ]:
(-ds_nino.T.differentiate('lat')*ds_nino.u).loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr')

In [ ]:
ds_nino.t.differentiate('lat').loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr')

In [ ]:
ds_nino.V.loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr')

In [ ]:
ds_nino.t.loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21, cmap='bwr')